In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
from nltk.tokenize import sent_tokenize
import json
from tqdm import tqdm
from pathlib import Path
import logging

# Create sentenceBERT embeddings

In [3]:
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Configuration ---

# Set path to project root
PROJECT_ROOT = Path("/Users/maxschaffelder/Desktop/Thesis")


# Model and sampling parameters
SENTENCE_TRANSFORMER_MODEL = "all-MiniLM-L6-v2"
SAMPLE_SIZE = 5000
RANDOM_SEED = 42

# Define input data paths relative to the project root
DATA_PATHS = {
    "Small": [
        "/Users/maxschaffelder/Desktop/Thesis/data/exp_1/small/outputs/output_lora_llama_8b_single.jsonl",
        "/Users/maxschaffelder/Desktop/Thesis/data/exp_1/small/outputs/output_lora_llama_8b_multi.jsonl",
        "/Users/maxschaffelder/Desktop/Thesis/data/exp_1/small/outputs/output_lora_llama_8b_human.jsonl",
        "/Users/maxschaffelder/Desktop/Thesis/data/finetuning/synthetic/Small/Llama/dolly_test_Llama.jsonl"
    ]
}

# Define output directory relative to the project root
OUTPUT_BASE_DIR = PROJECT_ROOT / "data" / "exp_1" / "small" / "sem_div" / "sentence_embeddings"


# --- Helper Functions ---

def is_valid_sentence(sentence: str) -> bool:
    """
    Check if a sentence is valid (not just a number or numbered list marker).
    """
    sentence = sentence.strip()
    if not sentence:
        return False
    
    cleaned = sentence.replace('#', '').replace('*', '').strip()
    if cleaned.replace('.', '', 1).isdigit():
        return False
    
    if len(cleaned) < 3:
        return False
        
    parts = cleaned.split('.')
    if len(parts) > 0 and parts[0].isdigit() and len(cleaned) < 5: # Catches "1. " etc.
        return False
    
    return True

def load_and_process_sentences(file_path: Path) -> list[str]:
    """
    Loads data from a .jsonl file, extracts responses, tokenizes them into
    sentences, and filters for valid sentences.
    """
    all_sentences = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                data = json.loads(line)
                response = data.get("response_model")
                if response:
                    sentences = sent_tokenize(response)
                    valid_sentences = [s for s in sentences if is_valid_sentence(s)]
                    all_sentences.extend(valid_sentences)
            except json.JSONDecodeError:
                logging.warning(f"Skipping malformed line in {file_path.name}")
    return all_sentences

def save_embeddings(output_path: Path, sentences: list[str], embeddings: np.ndarray, source_name: str):
    """
    Saves sentences and their embeddings to a .jsonl file.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for sentence, embedding in zip(sentences, embeddings):
            entry = {
                "sentence": sentence,
                "embedding": embedding.tolist(),
                "source": source_name
            }
            f.write(json.dumps(entry) + '\n')

# --- Main Processing Logic ---

def main():
    """
    Main function to generate and save sentence embeddings.
    """
    logging.info(f"Using project root: {PROJECT_ROOT}")
    logging.info(f"Loading sentence transformer model: {SENTENCE_TRANSFORMER_MODEL}")
    model = SentenceTransformer(SENTENCE_TRANSFORMER_MODEL)
    np.random.seed(RANDOM_SEED)

    for size, file_list in DATA_PATHS.items():
        logging.info(f"Processing {size} models...")
        output_dir_size = OUTPUT_BASE_DIR / size
        
        for file_path_str in tqdm(file_list, desc=f"Files for {size}"):
            file_path = PROJECT_ROOT / file_path_str
            model_name = file_path.stem
            
            if not file_path.exists():
                logging.warning(f"File not found: {file_path}. Skipping.")
                continue

            logging.info(f"Processing file: {file_path.name}")

            all_sentences = load_and_process_sentences(file_path)

            if len(all_sentences) >= SAMPLE_SIZE:
                sampled_indices = np.random.choice(len(all_sentences), size=SAMPLE_SIZE, replace=False)
                sampled_sentences = [all_sentences[i] for i in sampled_indices]
            else:
                logging.warning(f"Not enough sentences in {file_path.name} ({len(all_sentences)} found). Using all available.")
                sampled_sentences = all_sentences
            
            if not sampled_sentences:
                logging.warning(f"No valid sentences found in {file_path.name}. Skipping.")
                continue

            logging.info(f"  Calculating embeddings for {len(sampled_sentences)} sentences...")
            embeddings = model.encode(sampled_sentences, show_progress_bar=True)
            
            output_file = output_dir_size / f"{model_name}_embeddings.jsonl"
            save_embeddings(output_file, sampled_sentences, embeddings, model_name)
            logging.info(f"  Saved embeddings to {output_file}")

    logging.info("All embeddings have been generated and saved.")

if __name__ == "__main__":
    main()

2025-06-24 18:23:40,062 - INFO - Using project root: /Users/maxschaffelder/Desktop/Thesis
2025-06-24 18:23:40,063 - INFO - Loading sentence transformer model: all-MiniLM-L6-v2
2025-06-24 18:23:40,089 - INFO - Use pytorch device_name: mps
2025-06-24 18:23:40,090 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2025-06-24 18:23:42,366 - INFO - Processing Small models...
Files for Small:   0%|          | 0/4 [00:00<?, ?it/s]2025-06-24 18:23:42,369 - INFO - Processing file: output_lora_llama_8b_single.jsonl
2025-06-24 18:23:42,577 - INFO -   Calculating embeddings for 5000 sentences...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

2025-06-24 18:23:52,070 - INFO -   Saved embeddings to /Users/maxschaffelder/Desktop/Thesis/data/exp_1/small/sem_div/sentence_embeddings/Small/output_lora_llama_8b_single_embeddings.jsonl
Files for Small:  25%|██▌       | 1/4 [00:09<00:29,  9.70s/it]2025-06-24 18:23:52,072 - INFO - Processing file: output_lora_llama_8b_multi.jsonl
2025-06-24 18:23:52,249 - INFO -   Calculating embeddings for 5000 sentences...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

2025-06-24 18:23:59,317 - INFO -   Saved embeddings to /Users/maxschaffelder/Desktop/Thesis/data/exp_1/small/sem_div/sentence_embeddings/Small/output_lora_llama_8b_multi_embeddings.jsonl
Files for Small:  50%|█████     | 2/4 [00:16<00:16,  8.26s/it]2025-06-24 18:23:59,318 - INFO - Processing file: output_lora_llama_8b_human.jsonl
2025-06-24 18:23:59,394 - INFO -   Calculating embeddings for 5000 sentences...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

2025-06-24 18:24:06,181 - INFO -   Saved embeddings to /Users/maxschaffelder/Desktop/Thesis/data/exp_1/small/sem_div/sentence_embeddings/Small/output_lora_llama_8b_human_embeddings.jsonl
Files for Small:  75%|███████▌  | 3/4 [00:23<00:07,  7.62s/it]2025-06-24 18:24:06,182 - INFO - Processing file: dolly_test_Llama.jsonl
2025-06-24 18:24:06,377 - INFO -   Calculating embeddings for 5000 sentences...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

2025-06-24 18:24:13,123 - INFO -   Saved embeddings to /Users/maxschaffelder/Desktop/Thesis/data/exp_1/small/sem_div/sentence_embeddings/Small/dolly_test_Llama_embeddings.jsonl
Files for Small: 100%|██████████| 4/4 [00:30<00:00,  7.69s/it]
2025-06-24 18:24:13,124 - INFO - All embeddings have been generated and saved.
